In [1]:
import os
from google.colab import drive

# Hubungkan Google Colab ke Google Drive Anda
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import cv2
import glob
import shutil

def resize_yolo_dataset(
    base_dir,
    output_dir,
    target_w=640,
    target_h=640
):

    splits = ['train', 'valid', 'test']

    for split in splits:

        img_dir = os.path.join(base_dir, split, 'images')
        lbl_dir = os.path.join(base_dir, split, 'labels')

        # cek folder
        if not os.path.exists(img_dir):
            print(f"[-] Folder images {split} tidak ditemukan")
            continue

        print(f"\n[+] Memproses: {split.upper()}")

        # output folder
        out_img_dir = os.path.join(output_dir, split, 'images')
        out_lbl_dir = os.path.join(output_dir, split, 'labels')

        os.makedirs(out_img_dir, exist_ok=True)
        os.makedirs(out_lbl_dir, exist_ok=True)

        # ambil semua gambar
        image_paths = []

        extensions = [
            '*.jpg',
            '*.jpeg',
            '*.png',
            '*.JPG',
            '*.JPEG',
            '*.PNG'
        ]

        for ext in extensions:
            image_paths.extend(
                glob.glob(os.path.join(img_dir, ext))
            )

        total = 0

        for img_path in image_paths:

            filename = os.path.basename(img_path)
            basename = os.path.splitext(filename)[0]

            # path label
            lbl_path = os.path.join(
                lbl_dir,
                f"{basename}.txt"
            )

            # =========================
            # LOAD IMAGE
            # =========================

            img = cv2.imread(img_path)

            if img is None:
                print(f"[-] Gagal membaca: {filename}")
                continue

            # =========================
            # RESIZE IMAGE
            # =========================

            resized_img = cv2.resize(
                img,
                (target_w, target_h)
            )

            # simpan image
            out_img_path = os.path.join(
                out_img_dir,
                filename
            )

            cv2.imwrite(out_img_path, resized_img)

            # =========================
            # COPY LABEL
            # =========================

            out_lbl_path = os.path.join(
                out_lbl_dir,
                f"{basename}.txt"
            )

            # kalau label ada -> copy
            if os.path.exists(lbl_path):

                shutil.copy(
                    lbl_path,
                    out_lbl_path
                )

            else:
                # background image
                open(out_lbl_path, 'w').close()

            total += 1

        print(f"[✓] {total} gambar selesai diproses")


# ====================================
# PATH DATASET
# ====================================

INPUT_DATASET_DIR = "/content/drive/MyDrive/SIB/dataset-fix"

OUTPUT_DATASET_DIR = "/content/drive/MyDrive/SIB/preprocessing/dataset-resizing-baru"


# ====================================
# JALANKAN
# ====================================

resize_yolo_dataset(
    base_dir=INPUT_DATASET_DIR,
    output_dir=OUTPUT_DATASET_DIR,
    target_w=640,
    target_h=640
)

print("\n[SUKSES] Resize dataset selesai!")


[+] Memproses: TRAIN
[✓] 282 gambar selesai diproses

[+] Memproses: VALID
[✓] 81 gambar selesai diproses

[+] Memproses: TEST
[✓] 40 gambar selesai diproses

[SUKSES] Resize dataset selesai!


# Skip

In [ ]:

import cv2
import os
import glob

def resize_yolo_dataset(base_dir, output_dir, target_w=640, target_h=640):
    # Cari semua subfolder di dalam direktori utama (train, valid, test)
    splits = ['train', 'valid', 'test']

    for split in splits:
        img_dir = os.path.join(base_dir, split, 'images')
        lbl_dir = os.path.join(base_dir, split, 'labels')

        # Cek apakah folder ada
        if not os.path.exists(img_dir) or not os.path.exists(lbl_dir):
            print(f"[-] Folder {split} tidak lengkap atau dilewati.")
            continue

        print(f"\n[+] Memproses bagian: {split.upper()}")

        # Buat folder output untuk hasil resize
        out_img_dir = os.path.join(output_dir, split, 'images')
        out_lbl_dir = os.path.join(output_dir, split, 'labels')
        os.makedirs(out_img_dir, exist_ok=True)
        os.makedirs(out_lbl_dir, exist_ok=True)

        # Ambil semua file gambar (mendukung jpg, jpeg, png)
        img_extensions = ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']
        image_paths = []
        for ext in img_extensions:
            image_paths.extend(glob.glob(os.path.join(img_dir, ext)))

        for img_path in image_paths:
            filename = os.path.basename(img_path)
            basename = os.path.splitext(filename)[0]
            lbl_path = os.path.join(lbl_dir, f"{basename}.txt")

            # 1. Baca dan Resize Gambar
            img = cv2.imread(img_path)
            if img is None:
                continue
            orig_h, orig_w = img.shape[:2]

            resized_img = cv2.resize(img, (target_w, target_h))
            cv2.imwrite(os.path.join(out_img_dir, filename), resized_img)

            # 2. Proses File Label jika Ada
            out_lbl_path = os.path.join(out_lbl_dir, f"{basename}.txt")
            if os.path.exists(lbl_path):
                with open(lbl_path, 'r') as f:
                    lines = f.readlines()

                new_lines = []
                for line in lines:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        cls, x_c, y_c, w, h = parts

                        # Kembalikan koordinat YOLO (0-1) ke piksel absolut asli
                        abs_x = float(x_c) * orig_w
                        abs_y = float(y_c) * orig_h
                        abs_w = float(w) * orig_w
                        abs_h = float(h) * orig_h

                        # Hitung ulang koordinat YOLO berdasarkan ukuran target baru
                        new_x_c = abs_x / target_w
                        new_y_c = abs_y / target_h
                        new_w = abs_w / target_w
                        new_h = abs_h / target_h

                        # Pastikan nilai berada di rentang 0.0 - 1.0 (mencegah bug pembulatan)
                        new_x_c = min(max(new_x_c, 0.0), 1.0)
                        new_y_c = min(max(new_y_c, 0.0), 1.0)
                        new_w = min(max(new_w, 0.0), 1.0)
                        new_h = min(max(new_h, 0.0), 1.0)

                        new_lines.append(f"{cls} {new_x_c:.6f} {new_y_c:.6f} {new_w:.6f} {new_h:.6f}\n")

                # Simpan label baru
                with open(out_lbl_path, 'w') as f:
                    f.writelines(new_lines)
            else:
                # Jika gambar background (tanpa objek), buat file text kosong
                open(out_lbl_path, 'w').close()

        print(f"[v] Selesai memproses {len(image_paths)} gambar di {split}")

# --- KONFIGURASI JALUR FOLDER ---
# Ganti dengan path folder utama Anda
INPUT_DATASET_DIR = "/content/drive/MyDrive/SIB/dataset-fix"
OUTPUT_DATASET_DIR = "/content/drive/MyDrive/SIB/preprocessing/dataset-resizing"

# Jalankan fungsi pembesar/pengecil skala
resize_yolo_dataset(
    base_dir=INPUT_DATASET_DIR,
    output_dir=OUTPUT_DATASET_DIR,
    target_w=640,  # Lebar target baru
    target_h=640   # Tinggi target baru
)
print("\n[SUKSES] Semua dataset berhasil di-resize!")


[+] Memproses bagian: TRAIN
[v] Selesai memproses 282 gambar di train

[+] Memproses bagian: VALID
[v] Selesai memproses 81 gambar di valid

[+] Memproses bagian: TEST
[v] Selesai memproses 40 gambar di test

[SUKSES] Semua dataset berhasil di-resize!
